In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Relative Active Graph — one-cell launcher
# Run this cell to install deps, build Rust, and start the full app stack.
# Colab: public ngrok URLs are printed. Local: localhost URLs are printed.
# To stop everything, interrupt the kernel or call proc.terminate() on each entry in `procs`.
# ═══════════════════════════════════════════════════════════════════════════════
import os, sys, shutil, subprocess, time
from pathlib import Path

# ── Environment ───────────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = Path('/content/Relative_Active_Graph') if IN_COLAB else Path('.').resolve()

if IN_COLAB and not REPO_DIR.exists():
    os.system('git clone https://github.com/Eupham/Relative_Active_Graph.git /content/Relative_Active_Graph')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / 'lcs' / 'induction'))
sys.path.insert(0, str(REPO_DIR / 'lcs' / 'training'))
print(f'Repo: {REPO_DIR}')

# ── Install Python deps ───────────────────────────────────────────────────────
def pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# lcs/requirements.txt is space-separated on one line — split manually
lcs_pkgs = [
    tok
    for line in Path('lcs/requirements.txt').read_text().splitlines()
    for tok in line.split()
    if tok.strip()
]
pip(*lcs_pkgs)
pip('fastapi', 'uvicorn[standard]', 'pydantic>=2', 'pyngrok')
print('Python deps installed.')

# ── Build Rust engine ─────────────────────────────────────────────────────────
release_bin = REPO_DIR / 'target' / 'release' / 'csrre'
debug_bin   = REPO_DIR / 'target' / 'debug'   / 'csrre'

if release_bin.exists():
    print(f'Rust binary: {release_bin}')
elif debug_bin.exists():
    print(f'Rust binary (debug): {debug_bin}')
elif shutil.which('cargo'):
    print('Building Rust engine…')
    r = subprocess.run(['cargo', 'build', '--release'], cwd=str(REPO_DIR))
    if r.returncode != 0:
        subprocess.run(['cargo', 'build'], cwd=str(REPO_DIR), check=True)
    print('Rust build done.')
else:
    print('cargo not found — Rust engine unavailable; Python-only mode.')

# ── Install React deps ────────────────────────────────────────────────────────
npm_ok = shutil.which('npm') is not None
if npm_ok and (REPO_DIR / 'frontend' / 'package.json').exists():
    if not (REPO_DIR / 'frontend' / 'node_modules').exists():
        print('Installing React deps…')
        subprocess.check_call(['npm', 'install', '--prefix', 'frontend', '--silent'])
    print('React deps ready.')

# ── Launch services ───────────────────────────────────────────────────────────
env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([
    str(REPO_DIR / 'lcs' / 'training'),
    str(REPO_DIR / 'lcs' / 'induction'),
    env.get('PYTHONPATH', ''),
])

procs = []  # (label, Popen, port)

# FastAPI backend
procs.append(('FastAPI backend', subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'backend.server:app',
     '--host', '0.0.0.0', '--port', '8000', '--reload'],
    cwd=str(REPO_DIR), env=env,
    stdout=open('/tmp/backend.log', 'w'), stderr=subprocess.STDOUT,
), 8000))

# React dashboard (preferred) or Streamlit (fallback)
if npm_ok and (REPO_DIR / 'frontend' / 'node_modules').exists():
    fe_env = {**env, 'BROWSER': 'none', 'PORT': '3000'}
    procs.append(('React dashboard', subprocess.Popen(
        ['npm', 'start', '--prefix', 'frontend'],
        cwd=str(REPO_DIR), env=fe_env,
        stdout=open('/tmp/frontend.log', 'w'), stderr=subprocess.STDOUT,
    ), 3000))
else:
    procs.append(('Streamlit UI', subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.port', '8501', '--server.headless', 'true'],
        cwd=str(REPO_DIR), env=env,
        stdout=open('/tmp/streamlit.log', 'w'), stderr=subprocess.STDOUT,
    ), 8501))

time.sleep(3)  # let processes bind

# ── Print URLs ────────────────────────────────────────────────────────────────
print('\n' + '═' * 45)
if IN_COLAB:
    from pyngrok import ngrok
    print('  APP READY — PUBLIC URLS (Colab)')
    print('═' * 45)
    for label, _, port in procs:
        url = ngrok.connect(port).public_url
        print(f'  {label:22s}  {url}')
else:
    print('  APP READY — LOCAL URLS')
    print('═' * 45)
    for label, _, port in procs:
        print(f'  {label:22s}  http://localhost:{port}')
    print(f'\n  Logs: /tmp/backend.log  /tmp/frontend.log')
print('═' * 45)
print('To stop: [kernel] → Interrupt, or run:  [p.terminate() for _,p,_ in procs]')